In [0]:
# ============================================================
# Bronze — Source 13: S3 + Lambda (Image Metadata)
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=13_s3_lambda/
# Target: bronze.lambda catalog in Unity Catalog
#
# Tables:
#   - bronze.lambda.image_metadata
#
# Raw format: JSON (daily partitioned, Lambda event payloads)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "13_s3_lambda"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=18/")

In [0]:
# Cell 2 — Read all image event JSON files
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/hour=*/image_events_*.json"

df = spark.read.format("json") \
    .load(path) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"Raw rows: {df.count()}")
df.printSchema()

In [0]:
# Cell 3 — Flatten events array and write to Bronze
from pyspark.sql.functions import explode

events_flat = df \
    .select(explode(col("events")).alias("e")) \
    .select(
        col("e.event_id"),
        col("e.event_type"),
        col("e.event_source"),
        col("e.event_time"),
        col("e.product_sku") if "product_sku" in [f.name for f in df.select("events").schema.fields[0].dataType.elementType.fields] else lit(None).cast("string").alias("product_sku"),
        col("e.image_format"),
        col("e.image_position"),
        col("e.colour_space"),
        col("e.dpi"),
        col("e.height_px"),
        col("e.width_px") if "width_px" in [f.name for f in df.select("events").schema.fields[0].dataType.elementType.fields] else lit(None).cast("long").alias("width_px"),
        col("e.compression_ratio"),
        col("e.has_alpha"),
        col("e.has_white_background"),
        col("e.is_valid_format"),
        col("e.meets_min_size"),
    )

print(f"Flattened events: {events_flat.count()} rows")
events_flat.show(3)

In [0]:
# Cell 4 — Write to Bronze and update watermark
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.lambda")

events_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.lambda.image_metadata")

latest_ts = df.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, events_flat.count())
print(f"✅ bronze.lambda.image_metadata: {events_flat.count()} rows")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.lambda.image_metadata").collect()[0]['cnt']
print(f"bronze.lambda.image_metadata: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '13_s3_lambda'").show()